# Lab 4 — A knowledge worker over your own docs

**~60 minutes.** Ed's Week 5 project — an expert knowledge worker built on RAG — rebuilt
with **local embeddings**, so there is no API cost and no data leaving your laptop for the
retrieval half.

The corpus is this repository's own documentation: the concepts course and the Hermes
workshop docs. You will index it, ask it questions, and — the part most RAG tutorials
skip — **measure retrieval separately from generation**.

Rule to take away: when a RAG answer is wrong, check first whether the right chunk was even
retrieved. Teams lose weeks tuning prompts to fix a recall problem.

In [ ]:
import glob, json, re, textwrap
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

from shared import ask

DOCS = sorted(glob.glob("../../hermes-agent-workshop/docs/*.md")
              + glob.glob("../../presentation/*.md")
              + glob.glob("../../README.md"))
print(len(DOCS), "documents:")
for d in DOCS:
    print(f"  {Path(d).name:40} {len(open(d).read()):>7,} chars")

## 1. Chunking is where quality is won or lost

Too large and the embedding is diluted; too small and the meaning is severed from its
context. Fixed-size splitting is the lazy default — **structure-aware beats it**, and
markdown gives you the structure for free in the `##` headings.

Carry metadata on every chunk (source file, heading path). You need it for citations, for
filtering, and for the eval below.

In [ ]:
def chunk_markdown(path: str, max_chars: int = 1200) -> list[dict]:
    text = open(path).read()
    lines = text.split("\n")
    sections, heading, buf = [], Path(path).name, []

    def flush():
        body = "\n".join(buf).strip()
        if body:
            sections.append((heading, body))

    for line in lines:
        if line.startswith("#"):
            flush()
            heading = line.lstrip("#").strip()
            buf = []
        else:
            buf.append(line)
    flush()

    chunks = []
    for head, body in sections:
        if len(body) <= max_chars:
            pieces = [body]
        else:                                   # split long sections on blank lines
            pieces, current = [], ""
            for para in body.split("\n\n"):
                if len(current) + len(para) > max_chars and current:
                    pieces.append(current)
                    current = para
                else:
                    current = f"{current}\n\n{para}" if current else para
            if current:
                pieces.append(current)
        for piece in pieces:
            chunks.append({
                "text": f"[{Path(path).name} — {head}]\n{piece}",   # context header
                "source": Path(path).name,
                "heading": head,
            })
    return chunks


chunks = [c for path in DOCS for c in chunk_markdown(path)]
print(len(chunks), "chunks")
print(chunks[5]["heading"], "|", chunks[5]["text"][:200])

## 2. Embed and index

`all-MiniLM-L6-v2` runs on CPU in seconds and is good enough to learn on. The first run
downloads ~90 MB from huggingface.co — no token required, but it does need network.

**The rule that causes real incidents:** the same embedding model must be used for indexing
and for querying. Swap it and your index becomes silently meaningless.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma = chromadb.Client()
try:
    chroma.delete_collection("workshop")
except Exception:
    pass
collection = chroma.create_collection("workshop")

vectors = embedder.encode([c["text"] for c in chunks], show_progress_bar=True).tolist()
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=vectors,
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"], "heading": c["heading"]} for c in chunks],
)
print(collection.count(), "chunks indexed")

## 3. Retrieve, then answer — with citations

In [ ]:
def retrieve(question: str, k: int = 5) -> list[dict]:
    qvec = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=qvec, n_results=k)
    return [{"text": doc, "source": meta["source"], "heading": meta["heading"]}
            for doc, meta in zip(res["documents"][0], res["metadatas"][0])]


GROUNDED_SYSTEM = """Answer using ONLY the passages provided.

Cite the source for every claim, in the form (filename — heading).
If the passages do not contain the answer, say exactly what is missing and stop. Never fill
a gap from your own knowledge — an admitted gap is useful, an invented answer is not."""


def answer(question: str, k: int = 5) -> str:
    hits = retrieve(question, k)
    context = "\n\n---\n\n".join(
        f"({h['source']} — {h['heading']})\n{h['text']}" for h in hits)
    return ask(f"Passages:\n\n{context}\n\nQuestion: {question}",
               system=GROUNDED_SYSTEM, temperature=0.1)


print(answer("What is the difference between RAG and fine-tuning?"))

In [ ]:
for q in ["What ports does Hermes use?",
          "How should I choose a model for a task?",
          "What is the population of Denmark?"]:
    print(f"\n=== {q}\n{answer(q)}")

The third question is the important one. A grounded system should refuse it. If yours
answered from general knowledge, your system prompt is not doing its job — fix it now.

## 4. Measure retrieval on its own

Generation quality is capped by retrieval quality, so measure retrieval directly:
**recall@k** — for a set of questions with a known correct source, how often is that source
in the top k?

In [ ]:
EVAL = [
    ("What is a token and why does it matter for cost?", "ai-foundations-for-engineers.md"),
    ("How do I add web search to the agent?",            "tools.md"),
    ("What does the agentic loop look like step by step?", "basics-agents.md"),
    ("Which ports does the dashboard run on?",            "setup-guide.md"),
    ("How does temperature affect output?",               "basics-llms.md"),
    ("What should I do about prompt injection?",          "ai-foundations-for-engineers.md"),
    ("How do I process an invoice PDF by email?",         "use-cases.md"),
    ("What does the workshop cover in the first hour?",   "workshop-script.md"),
]


def recall_at_k(k: int = 5) -> float:
    hits = 0
    for question, expected in EVAL:
        sources = [h["source"] for h in retrieve(question, k)]
        ok = expected in sources
        hits += ok
        print(f"  {'ok ' if ok else 'MISS'} k={k} {question[:46]:48} -> {sources[:3]}")
    score = hits / len(EVAL)
    print(f"recall@{k} = {score:.0%}")
    return score


for k in (1, 3, 5):
    recall_at_k(k)
    print()

## 5. Improve it, then re-measure

Pick one change and prove it. Options, in rough order of payoff:

- **hybrid search** — add a keyword pass (simple word overlap is enough here) and merge the
  rankings. Keyword still wins on identifiers, error codes and exact names,
- **chunk size** — halve or double `max_chars` and rebuild,
- **query rewriting** — expand the question with a cheap model call before embedding.

Re-run `recall_at_k`. Keep both numbers.

In [ ]:
def keyword_scores(question: str) -> dict:
    terms = {w for w in re.findall(r"[a-z0-9]+", question.lower()) if len(w) > 3}
    scores = {}
    for i, c in enumerate(chunks):
        body = c["text"].lower()
        scores[i] = sum(body.count(t) for t in terms)
    return scores


def retrieve_hybrid(question: str, k: int = 5) -> list[dict]:
    """Vector top-2k merged with keyword top-2k, simple reciprocal-rank fusion."""
    qvec = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=qvec, n_results=2 * k)
    vec_ids = [int(i) for i in res["ids"][0]]

    kw = sorted(keyword_scores(question).items(), key=lambda kv: -kv[1])
    kw_ids = [i for i, s in kw[: 2 * k] if s > 0]

    fused = {}
    for rank, i in enumerate(vec_ids):
        fused[i] = fused.get(i, 0) + 1 / (60 + rank)
    for rank, i in enumerate(kw_ids):
        fused[i] = fused.get(i, 0) + 1 / (60 + rank)

    best = sorted(fused, key=lambda i: -fused[i])[:k]
    return [{"text": chunks[i]["text"], "source": chunks[i]["source"],
             "heading": chunks[i]["heading"]} for i in best]


def recall_hybrid(k: int = 3) -> float:
    hits = 0
    for question, expected in EVAL:
        sources = [h["source"] for h in retrieve_hybrid(question, k)]
        ok = expected in sources
        hits += ok
        print(f"  {'ok ' if ok else 'MISS'} {question[:46]:48} -> {sources[:3]}")
    print(f"hybrid recall@{k} = {hits/len(EVAL):.0%}")
    return hits / len(EVAL)


recall_hybrid(3)

## Stretch goals

1. **Break it on purpose.** Re-index with `max_chars=200` and again with `max_chars=6000`.
   Watch recall move. This is the single most instructive experiment in the lab.
2. **Permissions.** Add an `audience` field to the metadata ("internal" / "public") and
   filter at query time. Note that the model never sees filtered chunks — that is how access
   control works in RAG, and why it must happen at retrieval, never in the prompt.
3. **Visualise.** Reduce the embeddings to 2-D (t-SNE or UMAP) and plot them coloured by
   source file. Ed does this in his course, and it makes "similar things are nearby" concrete.
4. **Your own corpus.** Point `DOCS` at your team's runbooks and write eight eval questions
   with known sources. That artefact outlives this workshop.